In [ ]:
import os

In [ ]:
ld_library_path = os.environ.get("LD_LIBRARY_PATH", "") # Get current LD_LIBRARY_PATH (if set)
conda_lib_path = 'data/conda/envs/swift/lib:/lib/x86_64-linux-gnu'
new_ld_library_path = f"{conda_lib_path}:{ld_library_path}"
os.environ["LD_LIBRARY_PATH"] = new_ld_library_path # Set it

# model

In [ ]:
import sys
sys.path.append('workplace/RA/repo/Qwen2.5-VL/qwen-vl-utils/src/')

from qwen_vl_utils import process_vision_info

In [ ]:
import torch

from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor

In [ ]:
# We recommend enabling flash_attention_2 for better acceleration and memory saving, especially in multi-image and video scenarios.
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    torch_dtype=torch.float16,
    attn_implementation="flash_attention_2",
    device_map="auto",
)

# default processer
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")

# The default range for the number of visual tokens per image in the model is 4-16384.
# You can set min_pixels and max_pixels according to your needs, such as a token range of 256-1280, to balance performance and cost.
# min_pixels = 256*28*28
# max_pixels = 1280*28*28
# processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct", min_pixels=min_pixels, max_pixels=max_pixels)


# data

In [ ]:
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

In [ ]:
data_dir = '../data/'
synthetic_data_dir = '../synthetic_data/'

raw_report_dir = '../reports/rd_images/'
synthetic_report_dir = '../reports/dreambooth/synthetic_train_realistic/'
os.listdir(data_dir)

In [ ]:
disease_images_df = pd.read_csv(os.path.join(data_dir, 'disease_images.csv'))
disease_images_df.head()

In [ ]:
abbr_to_name = disease_images_df[['disease_abbr', 'disease_name']].drop_duplicates().set_index('disease_abbr')['disease_name'].to_dict()


# evaluate disease prediction acc

In [ ]:
from rapidfuzz import fuzz

def extract_assistant_part(text):
    marker = "assistant\n"
    if marker in text:
        return text.split(marker, 1)[1].strip()
    else:
        return ""  

def extract_diagnostic_part(text):
    marker = "Diagnostic Suggestion:"
    if marker in text:
        # print(text.split(marker, 1))
        return text.split(marker, 1)[1].strip()
    else:
        return "" 
    
def is_disease_predicted(disease_name, diagnostic_text, threshold=80):
    disease = disease_name.lower()
    text = diagnostic_text.lower()
    return fuzz.partial_ratio(disease, text) >= threshold

In [ ]:
raw_pred_correct_report_dir = '../results/raw_correct_report/'

cnt, cnt_disease = 0, 0
for raw_report_name in os.listdir(raw_report_dir):
    abbr = raw_report_name.split('.')[0]
    disease_name = abbr_to_name.get(abbr, None)
    with open(os.path.join(raw_report_dir, raw_report_name), "r", encoding="utf-8") as f:
        raw_report_text = f.read()

    assistant_text = extract_assistant_part(raw_report_text)
    diagnostic_text = extract_diagnostic_part(assistant_text)

    is_disease_correct = is_disease_predicted(disease_name, diagnostic_text)
    cnt_disease += 1
    if is_disease_correct:
        cnt += 1
        # write the correct report
        with open(os.path.join(raw_pred_correct_report_dir, raw_report_name), "w", encoding="utf-8") as f:
            f.write(raw_report_text)
        
# acc = 0.008771929824561403


In [ ]:
sys_pred_correct_report_dir = '../results/sys_correct_report/'

cnt, cnt_disease = 0, 0
for sys_report_name in os.listdir(synthetic_report_dir):
    abbr = sys_report_name.split('_')[0]
    disease_name = abbr_to_name.get(abbr, None)
    with open(os.path.join(synthetic_report_dir, sys_report_name), "r", encoding="utf-8") as f:
        sys_report_text = f.read()

    assistant_text = extract_assistant_part(raw_report_text)
    diagnostic_text = extract_diagnostic_part(assistant_text)

    is_disease_correct = is_disease_predicted(disease_name, diagnostic_text)
    cnt_disease += 1
    if is_disease_correct:
        cnt += 1
        # write the correct report
        with open(os.path.join(sys_pred_correct_report_dir, sys_report_name), "w", encoding="utf-8") as f:
            f.write(sys_report_text)
        
# acc = 0.00

In [ ]:
cnt, cnt_disease, cnt/cnt_disease

In [ ]:
abbr_to_name['CSY']

# evaluate similarity between raw & sys reports

In [ ]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# similarity function
def compute_tfidf_similarity(text1, text2):
    vec = TfidfVectorizer().fit_transform([text1, text2])
    return cosine_similarity(vec[0:1], vec[1:2])[0][0]

# region extractor
def extract_assistant_part(text):
    if "assistant" in text:
        return text.split("assistant", 2)[2].strip()
    return text.strip()

def extract_region_paragraph(text, region_name):
    pattern = rf"{re.escape(region_name)}:\s*(.*?)(?=\n[A-Z][a-zA-Z\s/]+:|\Z)"
    match = re.search(pattern, text, re.DOTALL)
    return match.group(1).strip() if match else ""

In [ ]:
similarity_results = []
regions = [
    "Left Eye",
    "Right Eye",
    "Nose",
    "Mouth/Lips",
]
all_sys_reports_name = os.listdir(synthetic_report_dir)

for raw_report_name in os.listdir(raw_report_dir):
    # print(raw_report_name)
    abbr = raw_report_name.split('.')[0]
    disease_name = abbr_to_name.get(abbr, None)
    with open(os.path.join(raw_report_dir, raw_report_name), "r", encoding="utf-8") as f:
        raw_report_text = f.read()
    raw_assistant_text = extract_assistant_part(raw_report_text)

    sys_reports_name = [name for name in all_sys_reports_name if name.startswith(abbr)]
    for sys_report_name in sys_reports_name:
        # print(raw_report_name, sys_report_name)
        with open(os.path.join(synthetic_report_dir, sys_report_name), "r", encoding="utf-8") as f:
            sys_report_text = f.read()
        sys_assistant_text = extract_assistant_part(sys_report_text)

        result = {
            "abbr": abbr,
            "disease_name": disease_name,
            "raw_report": raw_report_name,
            "sys_report": sys_report_name,
        }

        # overall similarity
        result["overall_similarity"] = round(compute_tfidf_similarity(raw_assistant_text, sys_assistant_text), 4)

        # each region similarity
        for region in regions:
            raw_region = extract_region_paragraph(raw_assistant_text, region)
            sys_region = extract_region_paragraph(sys_assistant_text, region)

            if raw_region and sys_region:
                sim = compute_tfidf_similarity(raw_region, sys_region)
            else:
                sim = 0.0
            result[f"{region}_similarity"] = round(sim, 4)

        similarity_results.append(result)



In [ ]:
len(similarity_results)

In [ ]:
similarity_df = pd.DataFrame(similarity_results)
# sorting by overall similarity
similarity_df = similarity_df.sort_values(by="overall_similarity", ascending=False)
# reset index
similarity_df = similarity_df.reset_index(drop=True)

In [ ]:
similarity_df

In [ ]:
similarity_df.to_csv('tfidf_similarity_results.csv', index=False)

# sentence similarity

In [ ]:
from sentence_transformers import SentenceTransformer, util

In [ ]:
# model_name = 'all-MiniLM-L6-v2'
# BioBERT
# model_name = 'dmis-lab/biobert-base-cased-v1.2'

model_name = 'pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb'
model = SentenceTransformer(model_name)


In [ ]:
def compute_semantic_similarity(text1, text2):
    emb1 = model.encode(text1, convert_to_tensor=True)
    emb2 = model.encode(text2, convert_to_tensor=True)
    return float(util.pytorch_cos_sim(emb1, emb2)[0][0])

In [ ]:
neg_phrases = [
    " normal",
    "no signs of",
    "no significant",
    "no evidence",
    "no abnormalities",
    "no visible",
]

def is_normal_region(text, neg_phrases):
    text = text.lower()
    return any(p in text for p in neg_phrases)

In [ ]:
similarity_results = []
regions = [
    "Left Eye",
    "Right Eye",
    "Nose",
    "Mouth/Lips",
]
all_sys_reports_name = os.listdir(synthetic_report_dir)

for raw_report_name in tqdm(os.listdir(raw_report_dir)):
    abbr = raw_report_name.split('.')[0]
    disease_name = abbr_to_name.get(abbr, None)
    with open(os.path.join(raw_report_dir, raw_report_name), "r", encoding="utf-8") as f:
        raw_report_text = f.read()
    raw_assistant_text = extract_assistant_part(raw_report_text)

    sys_reports_name = [name for name in all_sys_reports_name if name.startswith(abbr)]
    for sys_report_name in sys_reports_name:
        with open(os.path.join(synthetic_report_dir, sys_report_name), "r", encoding="utf-8") as f:
            sys_report_text = f.read()
        sys_assistant_text = extract_assistant_part(sys_report_text)

        result = {
            "abbr": abbr,
            "disease_name": disease_name,
            "raw_report": raw_report_name,
            "sys_report": sys_report_name,
        }

        # overall_similarity
        result["overall_similarity"] = round(compute_semantic_similarity(raw_assistant_text, sys_assistant_text), 4)

        normal_count = 0
        # region similarity
        for region in regions:
            raw_region = extract_region_paragraph(raw_assistant_text, region)
            sys_region = extract_region_paragraph(sys_assistant_text, region)

            if raw_region and sys_region:
                sim = compute_semantic_similarity(raw_region, sys_region)
            else:
                sim = 0.0

            result[f"{region}_similarity"] = round(sim, 4)

            if is_normal_region(sys_region, neg_phrases):
                normal_count += 1

        if normal_count > 0:
            continue
        print(f"raw report: {raw_report_name}, sys report: {sys_report_name}")

        similarity_results.append(result)

In [ ]:
similarity_df = pd.DataFrame(similarity_results)
similarity_df = similarity_df.sort_values(by="overall_similarity", ascending=False).reset_index(drop=True)
similarity_df.to_csv('BioBERT_filter' + '_semantic_similarity_results.csv', index=False)

In [ ]:
similarity_df = pd.read_csv('BioBERT' + '_semantic_similarity_results.csv')

In [ ]:
similarity_df

In [ ]:
# ouput TABLE SHOW average similarity +- std of each region
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

similarity_df = pd.read_csv('mpnet_semantic_similarity_results.csv')

In [ ]:
similarity_cols = [col for col in similarity_df.columns if col.endswith("_similarity")]
summary_stats = similarity_df[similarity_cols].agg(["mean", "std"]).T
summary_stats["mean ± std"] = summary_stats["mean"].round(4).astype(str) + " ± " + summary_stats["std"].round(4).astype(str)

In [ ]:
summary_stats